## Imports and Configuration

In [ ]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine,inspect
from urllib.parse import quote_plus

# --- CONFIGURATION --
SS_DB = {
    "server": "<host_name>",
    "user": "<user_name>",  # Your SQL Server username (e.g., 'sa')
    "pass": "<password>",
    "db": "<db_name>",
    "driver": "ODBC Driver 18 for SQL Server" 
}

PG_DB = {
    "user": "<user_name>",
    "pass": "<password>",
    "host": "<host_name>",
    "port": "5432",
    "db": "<db_name>"
}

print("Libraries imported and config set")

## Create Engines and Test Connections
- verifies that both databases are reachable before you start moving data

In [ ]:
# sql server connection string
ss_conn_str = (
    f"DRIVER={{{SS_DB['driver']}}};"
    f"SERVER={SS_DB['server']};"
    f"DATABASE={SS_DB['db']};"
    f"UID={SS_DB['user']};"
    f"PWD={SS_DB['pass']};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

ss_engine = create_engine(f"mssql+pyodbc:///?odbc_connect={ss_conn_str}")

# URL-encode the password to handle the '@' and '!' characters
pg_pass_encoded = quote_plus(PG_DB['pass'])

# Build the URL using the encoded password
pg_url = f"postgresql://{PG_DB['user']}:{pg_pass_encoded}@{PG_DB['host']}:{PG_DB['port']}/{PG_DB['db']}"

# Create the engine
pg_engine = create_engine(pg_url)


# Test Connections
try:
    with pg_engine.connect() as conn:
        print("PostgreSQL Connection Successful")
    with ss_engine.connect() as conn:
        print("SQL Server Connection Successful")
except Exception as e:
    print(f"Connection Error: {e}")

## Inspect Tables
- discover all tables in Postgres schema

In [ ]:
inspector = inspect(ss_engine)
ss_tables = inspector.get_table_names(schema='dbo')

print(f"Found {len(ss_tables)} tables in SQL Server: {ss_tables}")

## The Migration Loop
- iterates through the tables and provides a visual progress update

In [ ]:
for table in ss_tables:
    try:
        print(f"Reading {table}...")
        # Read from sql server database
        df = pd.read_sql_table(table, ss_engine, schema='dbo')

        if df.empty:
            print(f"Table {table} is empty. Skipping...")
            continue

        # append the prefix for the destination
        destination_name = f"stg_{table}"
        
        # Load to PG (if_exists='replace' creates the DDL/Table automatically)
        print(f"Writing {table} to PostgreSQL ({len(df)} rows)...")
        df.to_sql(
            destination_name, 
            pg_engine, 
            if_exists='replace', 
            index=False, 
            chunksize=1000
        )

        print(f"Successfully moved {table} -> {destination_name}")
        
    except Exception as e:
        print(f"Failed to migrate {table}: {e}")

print("\n--- ALL TASKS COMPLETE ---")